# 2.4 · 抽样方法 / Sampling Methods

> **课程定位**
> 2.3 的 SE 公式默认"简单随机抽样"。现实里怎么抽——**分层能白拿精度，整群会赔精度，抽错了全盘皆输（偏差不随 n 消失）**。本课也讲流式场景的水库抽样（经典面试编程题）。
> The SE formula assumes simple random sampling. Stratification buys free precision, clustering costs it, and biased sampling poisons everything — bias does NOT shrink with n.

> 💡 **面试相关**
> - "分层抽样为什么方差更小" ★★★★
> - "水库抽样算法" ★★★★（流式数据经典编程题）
> - "幸存者偏差举例" ★★★★（行为题+统计题双料）
> - "训练集怎么抽才不偏" ★★★

---

## 目录
1. [偏差 vs 方差：抽样的两种死法 ⭐](#1)
2. [简单随机抽样 SRS（基线）](#2)
3. [分层抽样：免费的精度 ⭐](#3)
4. [整群抽样：省钱但赔精度](#4)
5. [系统抽样与周期陷阱](#5)
6. [水库抽样 ⭐（流式 + 面试编程题）](#6)
7. [⚠ 抽样偏差动物园](#7)
8. [实战：合成人口普查对比四种抽法](#8)
9. [小结](#9)


<a id="1"></a>
## 1. 偏差 vs 方差：抽样的两种死法 ⭐

$$\mathbb{E}\big[(\hat{\theta} - \theta)^2\big] = \underbrace{\text{Bias}^2}_{\text{系统性偏}} + \underbrace{\text{Var}}_{\text{随机抖}}$$

| | 方差问题 | 偏差问题 |
|---|---|---|
| 来源 | 样本太少 | **抽样机制偏了** |
| 加大 n | ✅ 治好（$1/\sqrt n$）| ❌ **毫无帮助** |
| 例子 | 只问了 50 个人 | 只在高尔夫球场问收入 |

**1936 年文学文摘惨案**：237 万份问卷（超大 n！）预测兰登大胜罗斯福——错得离谱。样本来自电话簿和汽车注册（大萧条年代 = 富人），**n 巨大只是把错误答案估得更精确**。盖洛普用 5 万人的配额抽样答对了。
The Literary Digest polled 2.37M people and got it spectacularly wrong — biased frame (phone books in 1936 = the wealthy). Gallup won with 50K. **A huge n just estimates the wrong answer precisely.**


<a id="2"></a>
## 2. 简单随机抽样 SRS / Simple Random Sampling

每个个体**等概率、独立**入样——所有统计公式的"默认假设"。

$$\mathrm{SE}_{\text{SRS}}(\bar{x}) = \frac{\sigma}{\sqrt{n}} \times \underbrace{\sqrt{1 - \tfrac{n}{N}}}_{\text{有限总体修正 FPC}}$$

**FPC**：当抽样比 $n/N$ 不可忽略（>5%）时 SE 变小——抽到一半总体时你已经"半知道"答案了。$n/N \to 1$ 时 SE → 0（普查无误差）。
The finite-population correction kicks in when you sample a noticeable fraction; at a full census, SE hits zero.


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as st
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 造一个合成"城市人口收入普查" N=100,000，三个收入层
# Synthetic city census: 100K people in three income strata
N = 100_000
strata_spec = {                  # (占比, lognormal 参数)
    "low":    (0.55, (3.4, 0.35)),     # 中位数 ~30k
    "mid":    (0.35, (4.1, 0.30)),     # 中位数 ~60k
    "high":   (0.10, (5.0, 0.50)),     # 中位数 ~148k, 方差大
}
parts = []
for name, (frac, (m, s)) in strata_spec.items():
    inc = rng.lognormal(m, s, int(N*frac))
    parts.append(pd.DataFrame({"stratum": name, "income": inc}))
census = pd.concat(parts, ignore_index=True).sample(frac=1, random_state=0).reset_index(drop=True)

TRUE_MEAN = census["income"].mean()
print(census.groupby("stratum")["income"].agg(["count", "mean", "std"]).round(1))
print(f"\n真实总体均值 TRUE_MEAN = {TRUE_MEAN:.2f}k")


<a id="3"></a>
## 3. 分层抽样：免费的精度 ⭐ / Stratified Sampling

**思路**：按已知特征（收入层/地区/设备）把总体分层，**每层内部独立抽**，再加权合成。

$$\bar{x}_{\text{strat}} = \sum_h W_h\, \bar{x}_h, \qquad \mathrm{Var}(\bar{x}_{\text{strat}}) = \sum_h W_h^2\, \frac{\sigma_h^2}{n_h}$$

**为什么方差更小**：总方差 = 层内方差 + 层间方差。SRS 两种都要吃；分层抽样**层间方差直接清零**（每层配额固定，不会"这次碰巧多抽了富人"）。
Total variance = within-strata + between-strata. Stratification eliminates the between part entirely — no more "oops, this sample happened to catch extra rich people".

**配额怎么分**：
- **比例分配** / proportional：$n_h \propto N_h$ —— 简单，已经赢 SRS
- **Neyman 最优分配**：$n_h \propto N_h \sigma_h$ —— **方差大的层多抽**，精度最优


In [ ]:
def srs_sample(df, n): return df.sample(n)

def stratified_sample(df, n, neyman=False):
    if neyman:
        g = df.groupby("stratum")["income"].agg(["count", "std"])
        weights = g["count"] * g["std"]; weights /= weights.sum()
        alloc = (weights * n).round().astype(int)
    else:
        alloc = (df["stratum"].value_counts(normalize=True) * n).round().astype(int)
    return pd.concat([df[df.stratum == s].sample(k) for s, k in alloc.items()])

def stratified_mean(sample, pop_weights):
    # 加权合成 (权重 = 总体层占比, 不是样本占比)
    means = sample.groupby("stratum")["income"].mean()
    return sum(pop_weights[s] * means[s] for s in means.index)

pop_w = census["stratum"].value_counts(normalize=True).to_dict()

# 蒙特卡洛对比三种抽法的 SE / Monte Carlo: SE of three designs
n, n_sim = 500, 3000
results = {"SRS": [], "Stratified(比例)": [], "Stratified(Neyman)": []}
for _ in range(n_sim):
    results["SRS"].append(srs_sample(census, n)["income"].mean())
    results["Stratified(比例)"].append(stratified_mean(stratified_sample(census, n), pop_w))
    results["Stratified(Neyman)"].append(stratified_mean(stratified_sample(census, n, neyman=True), pop_w))

print(f"{'design':<22} {'bias':>8} {'SE':>8}   相对 SRS")
print("-" * 52)
se_srs = np.std(results["SRS"], ddof=1)
for k, v in results.items():
    bias, se = np.mean(v) - TRUE_MEAN, np.std(v, ddof=1)
    print(f"{k:<24} {bias:>8.3f} {se:>8.3f}   {se/se_srs:>6.1%}")


**同样 n=500，分层把 SE 砍掉一截，Neyman 再砍**（high 层方差最大 → 多抽它）。**等价说法：同样精度，分层可以省 20-40% 样本** = 真金白银的问卷费/标注费。

> 💡 ML 对应物：`train_test_split(stratify=y)`——类别不平衡时**保证 train/test 类比例一致**，就是分层抽样（Part 3.10 再见它）。
> The ML twin: `stratify=y` in train_test_split — same math, guaranteeing class proportions.


<a id="4"></a>
## 4. 整群抽样：省钱但赔精度 / Cluster Sampling

**场景**：调查全国学生——SRS 要飞遍 3000 所学校。**整群**：随机抽 30 所学校，**校内全测**。差旅省 100 倍。

**代价**：同校学生相似（同师资/生源）→ 有效信息少于名义 n。**设计效应**：
$$\mathrm{DEFF} = 1 + (m - 1)\,\rho$$
$m$ = 群内样本量，$\rho$ = **群内相关系数 ICC**。$\rho=0.1, m=50$ → DEFF ≈ 5.9：**5000 名义样本 ≈ 850 有效样本**。

| | 分层 / Stratified | 整群 / Cluster |
|---|---|---|
| 抽法 | 每层**都**抽一些 | 抽**部分**群，群内全取 |
| 对精度 | **更好**（消层间方差）| **更差**（群内冗余）|
| 动机 | 精度 | **省成本** |


In [ ]:
# 模拟整群抽样的精度损失 / Simulate the cluster penalty
# 200 个"社区"，社区间收入有差异 (ICC > 0)
n_clusters, cluster_size = 200, 500
cluster_effect = rng.normal(0, 15, n_clusters)            # 社区固定效应 → ICC
cluster_ids = np.repeat(np.arange(n_clusters), cluster_size)
income2 = 60 + cluster_effect[cluster_ids] + rng.normal(0, 25, n_clusters*cluster_size)
TRUE2 = income2.mean()

icc = cluster_effect.var() / (cluster_effect.var() + 25**2)
print(f"ICC ≈ {icc:.3f}")

n_sim, m = 2000, 50                                       # 每群抽 50 人
srs_means, cluster_means = [], []
for _ in range(n_sim):
    srs_means.append(rng.choice(income2, 500).mean())                       # SRS: 500 人
    chosen = rng.choice(n_clusters, 10, replace=False)                       # 整群: 10 群 × 50 人 = 500
    mask = np.isin(cluster_ids, chosen)
    idx = np.where(mask)[0]
    cluster_means.append(income2[rng.choice(idx, 500)].mean())

se_s, se_c = np.std(srs_means, ddof=1), np.std(cluster_means, ddof=1)
print(f"SRS SE      = {se_s:.3f}")
print(f"Cluster SE  = {se_c:.3f}   (赔了 {se_c/se_s:.1f}×)")
print(f"理论 DEFF 预测: √(1+(m-1)ρ) = {np.sqrt(1+(m-1)*icc):.1f}×")


<a id="5"></a>
## 5. 系统抽样与周期陷阱 / Systematic Sampling

**每隔 $k = N/n$ 个取一个**（随机起点）。流水线/名单场景实现极简，**通常 ≈ SRS**。

⚠ **致命陷阱：数据有周期**且周期 ≈ $k$ 时全盘崩溃。例：按小时记录的流量，$k=24$ → 每次都抽到同一时刻——"凌晨 3 点的流量"被当成全天均值。**抽之前看排序键有没有周期**。
Systematic sampling dies when the data has periodicity matching the step k — sampling hourly data with k=24 gives you 3 a.m. forever.


<a id="6"></a>
## 6. 水库抽样 ⭐ / Reservoir Sampling

**问题**：数据流式到达，**总量未知**（可能无限），内存只够存 $k$ 条——怎么保证最终留下的 $k$ 条是**均匀随机样本**？

**Algorithm R**：
1. 前 $k$ 条直接进"水库"
2. 第 $i$ 条（$i > k$）：以概率 $k/i$ 留下，随机顶掉水库中一条

**正确性证明**（归纳法核心步）：处理完第 $i$ 条后，任意已见元素在水库的概率 = $k/i$。
- 第 $i$ 条本身：留下概率 $k/i$ ✓
- 旧元素：$\underbrace{\tfrac{k}{i-1}}_{\text{原本在}} \times \big(1 - \underbrace{\tfrac{k}{i}\cdot\tfrac{1}{k}}_{\text{被顶掉}}\big) = \tfrac{k}{i-1}\cdot\tfrac{i-1}{i} = \tfrac{k}{i}$ ✓


In [ ]:
def reservoir_sample(stream, k, rng):
    # Algorithm R — O(n) 时间, O(k) 内存
    reservoir = []
    for i, x in enumerate(stream, start=1):
        if i <= k:
            reservoir.append(x)
        else:
            j = rng.integers(0, i)        # ∈ [0, i-1], P(j < k) = k/i
            if j < k:
                reservoir[j] = x
    return reservoir

# 验证均匀性: 流 0..999, k=10, 重复 20,000 次, 每个元素应入选 ~1%
counts = np.zeros(1000)
for _ in range(20_000):
    for x in reservoir_sample(range(1000), 10, rng):
        counts[x] += 1
freq = counts / 20_000

print(f"理论入选率 = k/n = 1.0%")
print(f"实测: mean={freq.mean():.4f}, min={freq.min():.4f}, max={freq.max():.4f}")
print(f"卡方均匀性 p = {st.chisquare(counts).pvalue:.3f}  (>0.05 = 均匀 ✓)")


**早到晚到入选率全是 1%** —— 均匀性成立。

> 💡 这是**字节/Google 高频编程题**。加分项：说出加权版（A-Res：键 $u^{1/w}$ 取 top-k）和分布式版（各分片独立水库再合并）。Spark 的 `sample()` 内部同款思想。


<a id="7"></a>
## 7. ⚠ 抽样偏差动物园 / The Bias Zoo

| 偏差 | 机制 | 经典案例 |
|---|---|---|
| **选择偏差** | 抽样框 ≠ 总体 | 文学文摘（电话簿=富人）|
| **幸存者偏差** ⭐ | 只观察到"活下来的" | **Wald 的轰炸机**：弹孔多的部位恰恰**不用**加固——被打中那里还能飞回来；打中没弹孔处（引擎）的没回来 |
| **无应答偏差** | 谁回答问卷不随机 | 满意度调查：极端不满者最爱填 |
| **自选择偏差** | 自己决定进组 | "上补习班的孩子成绩好"——报班家庭本就不同（→ Part 19 因果推断）|
| **长度偏差** | 入样概率 ∝ 持续时长 | 截面调查住院病人 → 重症（住得久）被过采 |

**共同点：n 越大错得越自信**。下面量化演示无应答偏差：


In [ ]:
# 无应答偏差: 满意度调查 / Non-response bias in a satisfaction survey
satisfaction = rng.normal(7.0, 1.5, 50_000).clip(0, 10)       # 真实均值 7.0

# 响应概率: 越不满越爱发声 / The dissatisfied respond far more
p_respond = np.clip(0.40 - 0.045 * satisfaction, 0.02, 1.0)   # 0 分者 40%, 8 分者 4%
responded = satisfaction[rng.random(50_000) < p_respond]

print(f"真实均值     = {satisfaction.mean():.3f}")
print(f"问卷均值     = {responded.mean():.3f}   (n={len(responded):,}, 偏 {responded.mean()-satisfaction.mean():+.2f})")
print(f"再收 10 倍问卷？偏差一分不少——这就是偏差 ≠ 方差")


<a id="8"></a>
## 8. 实战：四种抽法终极对比 / Final Shoot-out

同一总体、同样 n=500、各 3000 次重复——一张图看四种设计的抽样分布。


In [ ]:
n, n_sim = 500, 3000
designs = {"SRS": [], "Stratified": [], "Cluster": [], "Biased (rich 2x)": []}

# 偏置设计: 高收入层 2 倍入样概率 (模拟"商场拦访")
w_biased = np.where(census["stratum"] == "high", 2.0, 1.0)
w_biased /= w_biased.sum()

cl_label = (np.arange(len(census)) // 500) % 100              # 人造 100 个"社区"
census2 = census.assign(cluster=cl_label)

for _ in range(n_sim):
    designs["SRS"].append(census.sample(n)["income"].mean())
    designs["Stratified"].append(stratified_mean(stratified_sample(census, n), pop_w))
    chosen = rng.choice(100, 5, replace=False)
    designs["Cluster"].append(census2[census2.cluster.isin(chosen)].sample(n)["income"].mean())
    designs["Biased (rich 2x)"].append(census.sample(n, weights=w_biased)["income"].mean())

fig, ax = plt.subplots(figsize=(10, 4))
for name, vals in designs.items():
    ax.hist(vals, bins=70, density=True, alpha=0.55, label=f"{name}  (SE={np.std(vals, ddof=1):.2f})")
ax.axvline(TRUE_MEAN, color="k", ls="--", lw=2, label=f"TRUE = {TRUE_MEAN:.1f}")
ax.legend(fontsize=9); ax.set_xlabel("estimated mean income (k$)")
ax.set_title("Same n=500 — four designs, four sampling distributions")
plt.tight_layout(); plt.show()


**一图四个教训**：
1. **Stratified** 最窄且对中 —— 免费精度
2. **SRS** 无偏但稍宽 —— 基线
3. **Cluster** 最宽 —— 省钱的代价
4. **Biased** 整体右移 —— **唯一不对中的**，重复几次都救不回来

设计排序：偏差 > 设计选择 > 样本量。**先保证无偏，再谈精度**。


<a id="9"></a>
## 9. 小结 / Summary

```
MSE = Bias² + Var
  ├── Var: 加 n 可治 (1/√n)
  └── Bias: 加 n 无解 ⭐ — 文学文摘 237 万人翻车

设计谱系 (同 n 的精度):
  Stratified(Neyman) > Stratified(比例) > SRS ≈ Systematic > Cluster
  动机:    精度 ←─────────────────────────→ 成本

特殊场景:
  流式未知总量 → 水库抽样 (P(入选)=k/i, 归纳可证)
  ML 不平衡分类 → stratify=y (同款数学)
```

### 💡 面试速查
1. **分层为什么准**：消灭层间方差；Neyman 按 $N_h\sigma_h$ 配额
2. **DEFF** $= 1+(m-1)\rho$：整群的精度账单
3. **水库抽样**：第 $i$ 条以 $k/i$ 留下 + 归纳证明
4. **幸存者偏差标准故事**：Wald 加固没弹孔的引擎舱
5. **大 n 治不了偏差**——只会更自信地错

### 下一节
**2.5 置信区间**——SE 终于派上正式用场：$\bar x \pm t \cdot \mathrm{SE}$，以及"95% 置信"到底什么意思（多数人都答错）。
